# M4 · Inside the machine 3: feed-forward and the residual add

**Tiny Transformer, episode M4.** Attention moved information between the rows. Here you run the block's second half on the
line *“You taught me language; and my p”*: a second LayerNorm, each row widened from 28 numbers to 56 on its own, the GELU bend, back down to 28, and
the result added onto the stream. Then you measure what the whole block changed, by reading out the stream before and after it.

[Open in Google Colab](https://colab.research.google.com/github/gkiri/tiny_repo/blob/main/tiny-transformer/notebooks/M4-inside-feed-forward.ipynb) (execution uses Google's hosted runtime; no graphics card needed). On your own computer:
`pip install -r requirements.txt`, then `jupyter notebook M4-inside-feed-forward.ipynb`.

Every number here is computed in this notebook by the trained model: the course's model code with the recorded run's
9,417 learned numbers (`prompter.json`, step 98,000). The numbers the course's page shows are
**Recorded** in the course's export of this line; each cell checks that its own result matches the record to within 0.0002.

Laid out like the page: two sections, four chapters, and under each chapter its page's three steps (Picture it, Watch it on
our line, Read and run the code), then the code to run and a "your turn".

In [1]:
#@title Setup: run me first (one click) { display-mode: "form" }
# In Google Colab this fetches the course's labs (notebooks, data and the trained model) and moves into this notebook's
# folder, so every file is where the cells below look for it. On your own computer, inside a copy of the labs, it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/tiny_repo"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gkiri/tiny_repo", "/content/tiny_repo"], check=True)
    os.chdir("/content/tiny_repo/tiny-transformer/notebooks")
print("ready:", os.path.basename(os.getcwd()))

ready: notebooks


In [2]:
import base64, hashlib, json, os, urllib.request
import numpy as np
import torch

torch.manual_seed(0)
torch.set_grad_enabled(False)      # nothing is trained here, so no gradients are needed
print(torch.__version__)

2.10.0+cpu


### Setup: the text, the model and its trained numbers

The text gives the 65 characters their ids. The next cell is the model's whole code, unchanged; the one after it loads the
trained numbers into it.

In [3]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
here = os.path.abspath('')
candidates = [os.path.join(here, '..', 'data', 'tinyshakespeare.txt'),                                   # in the labs
              os.path.join(here, '..', '..', '..', '..', 'training', 'prompter', 'data', 'tinyshakespeare.txt')]   # in the course
found = [p for p in candidates if os.path.exists(p)]
raw = open(found[0], 'rb').read() if found else urllib.request.urlopen(URL, timeout=60).read()
assert hashlib.sha256(raw).hexdigest() == SHA256, "not the pinned Tiny Shakespeare"
text = raw.decode('utf-8')
print(f"{len(text):,} characters")

1,115,394 characters


In [4]:
"""Our tiny transformer: 9,417 learned numbers that predict the next character of Shakespeare.

It reads up to 32 characters of text and gives a probability to each of the 65 characters that could come next. Writing is
that prediction in a loop: pick a character, append it, slide the 32-character window, and predict again. (The class is
called Prompter for the course's file names; learners meet it as "our tiny transformer".)

The architecture is the "10k" configuration of maxpolaczuk/tiny-character-transformer (a 1-block, 2-head character GPT on Tiny
Shakespeare). The code here is written fresh for the course (that repository has no licence, so none of its code or weights
are copied): every line is one the course teaches.

    characters ─► token table (65 × 28) + position table (32 × 28)        lookup:      what, and where
               ─► one block: LayerNorm ─► 2-head causal attention ─► +    look back:   which earlier letters matter
                             LayerNorm ─► feed-forward 28 → 56 → 28 ─► +  think:       transform each row on its own
               ─► final LayerNorm ─► readout with the token table again   score all 65 characters
               ─► softmax                                                  probabilities for the next character

Parameters: token table 1,820 · positions 896 · block 6,580 (two LayerNorms 112, q|k|v 2,436, attention out 812,
feed-forward 1,624 + 1,596) · final LayerNorm 56 · readout bias 65 = 9,417.
"""

import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class Config:
    vocab_size: int = 65    # every distinct character in Tiny Shakespeare
    block_size: int = 32    # how many characters the model can read at once (its context)
    d_model: int = 28       # how many numbers describe one character in one place (a row)
    n_head: int = 2         # attention heads, each looking with 28 / 2 = 14 numbers
    d_ff: int = 56          # the feed-forward network's hidden width (2 × d_model)
    n_layer: int = 1        # transformer blocks

    def to_dict(self):
        return asdict(self)


class Tokenizer:
    """One id per character, in sorted order: '\\n' is 0, ' ' is 1, … 'z' is 64."""

    def __init__(self, text: str):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}

    def encode(self, s: str) -> list[int]:
        return [self.stoi[c] for c in s]

    def decode(self, ids) -> str:
        return "".join(self.chars[int(i)] for i in ids)


class CausalSelfAttention(nn.Module):
    """Each row asks a question (q), every earlier row offers a label (k) and a message (v). A row reads the messages of the
    rows whose labels match its question best, never a row to its right."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.n_head, self.head_dim = cfg.n_head, cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)     # q, k and v for both heads in one matrix
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)        # mix the heads' results back into one row
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("mask", mask.view(1, 1, cfg.block_size, cfg.block_size), persistent=False)

    def forward(self, x, keep=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)          # how well each question matches each label
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))   # no peeking at the future
        att = F.softmax(att, dim=-1)                                          # shares that add up to 1
        if keep is not None:
            keep["attn"] = att.detach()
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)            # each row: a weighted mix of messages
        return self.proj(y)


class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(), nn.Linear(cfg.d_ff, cfg.d_model))

    def forward(self, x, keep=None):
        x = x + self.attn(self.ln1(x), keep)    # look back, and add what was found to the row
        x = x + self.ff(self.ln2(x))            # think about each row on its own, and add that too
        return x


class Prompter(nn.Module):
    def __init__(self, cfg: Config = Config()):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)    # one learned row per character
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)    # one learned row per place 0 … 31
        self.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head_bias = nn.Parameter(torch.zeros(cfg.vocab_size))  # the readout reuses tok_emb (tied); only its bias is new
        self.apply(self._init)

    @staticmethod
    def _init(m):
        # small random numbers (σ = 0.02), so the untrained model's guesses are nearly even: its first loss is about
        # ln 65 = 4.17, the loss of a uniform guess over 65 characters
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, keep=None):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))   # what + where
        for b in self.blocks:
            x = b(x, keep)
        x = self.ln_f(x)
        logits = x @ self.tok_emb.weight.T + self.head_bias                         # a score for every character
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n, temperature=1.0, generator=None):
        for _ in range(n):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1, generator=generator)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [5]:
paths = [os.path.join(here, '..', 'model', 'prompter.json'),                                                     # in the labs
         os.path.join(here, '..', '..', '..', '..', 'src', 'specimen', 'tiny-transformer', 'data', 'prompter.json')]   # in the course
found = [p for p in paths if os.path.exists(p)]
assert found, "the model's trained numbers are not here: run the Setup cell at the top"
exp = json.load(open(found[0]))
state = {t['name']: torch.tensor(np.frombuffer(base64.b64decode(t['data']), dtype='<f4').reshape(t['shape']).copy()) for t in exp['weights']['tensors']}
tok = Tokenizer(text)
model = Prompter(Config(vocab_size=len(tok.chars)))
model.load_state_dict(state)
model.eval()
block = model.blocks[0]                            # the model's one block
print('trained for', f"{exp['checkpoint_step']:,}", 'steps:', f"{count_params(model):,}", 'learned numbers')
assert exp['checkpoint_step'] == 98000 and count_params(model) == 9417


def same(computed, recorded, tol=2e-4):
    """True when every computed number is within tol of the course's recorded one (the record keeps 4 decimals)."""
    a = torch.as_tensor(computed, dtype=torch.float32).flatten()
    b = torch.tensor(recorded, dtype=torch.float32).flatten()
    return a.shape == b.shape and bool((a - b).abs().max() <= tol)

trained for 98,000 steps: 9,417 learned numbers


The episode's teaching code, `code/inside_feed_forward.py`: each step of the block's second half, written out on the model's
own parts. Every comment gives a shape.

In [6]:
"""M4 · Inside the machine 3: feed-forward and the residual add (Tiny Transformer, ADR 0011; docs/BEGINNER-STANDARD.md §5).

The block's second half, written out on top of our tiny transformer's own parts (training/prompter/model.py): a second
LayerNorm and the first feed-forward layer widen each row from 28 numbers to 56, GELU bends every hidden number, the second
layer narrows each row back to 28, and the result is added onto the stream (the residual add). readout_at runs the model's
last stages on the stream at any point, to measure what the rows would predict there. Every function takes rows [T, 28]
(T = 32 for the running line) and treats each row on its own, with the same learned numbers for every row.

test_inside_feed_forward.py checks each function against model.py's own forward pass and against the recorded stages of the
running line (src/specimen/tiny-transformer/data/prompter-stages.json). The companion notebook, M4-inside-feed-forward.ipynb,
carries this file unchanged.
"""
import math

import torch


def widen(model, x):
    """A LayerNorm, then the first feed-forward layer: rows [T, 28] in, hidden numbers [T, 56] out (1,624 learned)."""
    block = model.blocks[0]                       # the model's one block
    up = block.ff[0]                              # the widening layer: weight [56, 28], bias [56]
    rows = block.ln2(x)                           # [T, 28]: each row to average 0 and spread 1, then a learned scale and shift
    return rows @ up.weight.T + up.bias           # [T, 28] @ [28, 56] + [56] → [T, 56]: the same weights for every row


def gelu(h):
    """The bend, number by number: GELU(h) = h × Φ(h), where Φ(h) is the fraction of the normal bell curve below h."""
    fraction = 0.5 * (1 + torch.erf(h / math.sqrt(2)))   # same shape as h: near 0 for very negative h, near 1 for large h
    return h * fraction                                  # [T, 56] → [T, 56]: negatives shrink toward 0, positives pass


def narrow(model, g):
    """The second feed-forward layer: bent hidden numbers [T, 56] in, an update for each row [T, 28] out (1,596 learned)."""
    down = model.blocks[0].ff[2]                  # the narrowing layer: weight [28, 56], bias [28]
    return g @ down.weight.T + down.bias          # [T, 56] @ [56, 28] + [28] → [T, 28]


def feed_forward(model, x):
    """The block's second half: widen, bend, narrow, then add the result onto the stream. [T, 28] in, [T, 28] out."""
    update = narrow(model, gelu(widen(model, x)))   # [T, 28]: what the feed-forward network writes for each row
    return x + update                               # [T, 28]: added, not swapped in: the old row stays in the sum


def readout_at(model, x):
    """The model's last stages, run on the stream at any point: what its rows would predict there. [T, 28] in, [T, 65] out."""
    rows = model.ln_f(x)                                        # [T, 28]: the final LayerNorm
    scores = rows @ model.tok_emb.weight.T + model.head_bias    # [T, 28] @ [28, 65] + [65] → [T, 65]: a score per character
    return torch.softmax(scores, dim=-1)                        # [T, 65]: probabilities, each row adds up to 1

### Where M3 left off

The line becomes 32 ids, then rows: token rows plus position rows, `x0` (M2). Attention adds its update, giving `x1` (M3). Both
are [32×28].

In [7]:
LINE = "You taught me language; and my p"
ids = torch.tensor(tok.encode(LINE))                                  # [32]
x0 = model.tok_emb(ids) + model.pos_emb(torch.arange(len(ids)))       # [32, 28]: what + where (M2)
att = block.attn(block.ln1(x0)[None])[0]                              # [32, 28]: attention's update (M3)
x1 = x0 + att                                                         # [32, 28]: added onto the stream
print('ids', tuple(ids.shape), ' x0', tuple(x0.shape), ' x1', tuple(x1.shape))
assert same(x0[31, -4:], [0.2553, 0.9579, -0.5443, 0.166]) and same(x1[31, -4:], [0.0533, 0.7352, -0.4811, 0.7264])      # row 31's last 4 numbers, as recorded

ids (32,)  x0 (32, 28)  x1 (32, 28)


# Section 1 · Each row alone

Attention moved information between the rows. Here each row works on what it gathered, on its own: widened by a learned layer,
then bent. Each chapter below follows its page: Picture it, Watch it on our line, then Read and run the code.

## M4.1 · Widen: each row from 28 numbers to 56

*What does the feed-forward network do to one row?*

**Predict first** (the page asks it too, before its clip): our widening layer turns each row's 28 numbers into 56 hidden
numbers. How many learned numbers does it hold: 1,568, 1,624 or 3,220?

### M4.1.1 · Picture it

After attention, each row holds what it gathered; to work on it, a copy gets more numbers. Take a made-up row of 2 numbers:
a hidden number is a weighted sum of both, plus a bias. Make 3 hidden numbers, each with its own weights and bias: 2 × 3
weights plus 3 biases, 9 learned numbers. Each hidden number checks the row for a pattern: the closer the match, the
larger the number. (The page's row is made up to show the idea.)

### M4.1.2 · Watch it on our line

*(The page's recorded clip, in words.)*

The rows come in from attention: `x1`, [32×28]. A second **LayerNorm** prepares a copy of them (56 learned numbers). Then
the first feed-forward layer widens each row from 28 numbers to 56: a [56×28] weight and 56 biases, 1,624 learned
numbers. The result is the **hidden** tensor, [32×56]; of row 31's 56 hidden numbers, 41 are negative.
Every row goes through the same weights, and each row goes alone: no row sees another here.

### M4.1.3 · Read and run the code

Widening is `widen` in `code/inside_feed_forward.py` (copied in above): it finds the block's widening layer, a [56×28]
weight and 56 biases; LayerNorm 2 prepares a copy of the rows; one line multiplies the copy by the weight and adds the bias.
The first cell runs it on our line and counts 1,624 learned numbers; the second changes row 30 alone, and only row 30 of the
hidden numbers moves.

In [8]:
up = block.ff[0]                                   # the widening layer
print('LayerNorm:', sum(p.numel() for p in block.ln2.parameters()), 'learned numbers')
print('widening layer: weight', tuple(up.weight.shape), '+ bias', tuple(up.bias.shape), '=', up.weight.numel() + up.bias.numel(), 'learned numbers')
hidden = widen(model, x1)                          # [32, 28] → [32, 56]
print('x1', tuple(x1.shape), '→ hidden', tuple(hidden.shape))
print("row 31's first 6 hidden numbers:", [round(v, 2) for v in hidden[31, :6].tolist()])
print("row 31's hidden numbers below 0:", int((hidden[31] < 0).sum()), 'of', hidden.shape[1])

RECORDED_HIDDEN_31 = [0.9293, -0.3947, -3.8487, 2.6411, 0.8741, -3.575, -1.1303, -2.1725, -8.0458, 0.6793, -1.4665, -0.5009, -0.9561, -1.5482, -5.9057, -1.017, -0.3269, -1.3339, -0.6843, -0.7366, -3.5557, 0.825, -2.2109, -3.4815, -8.8775, -3.166, -5.2941, -1.1989, -0.8689, 1.3175, -1.115, 1.2417, -2.7466, 1.2855, 3.1205, -4.363, 0.7775, -0.6255, 0.3344, -9.2259, -0.7418, -4.2001, -3.2686, -7.1553, -2.1384, 0.7221, -0.232, 1.3476, -1.0658, -6.1717, 1.065, -0.7396, -7.3999, -2.2145, -6.009, 3.576]   # the course's record of row 31's 56 hidden numbers
assert sum(p.numel() for p in block.ln2.parameters()) == 56 and up.weight.numel() + up.bias.numel() == 1624
assert tuple(hidden.shape) == (32, 56)
assert same(block.ln2(x1)[31, -4:], [-0.1136, 1.8061, -1.2529, 0.9633]) and same(hidden[31], RECORDED_HIDDEN_31)
assert int((hidden[31] < 0).sum()) == 41

LayerNorm: 56 learned numbers
widening layer: weight (56, 28) + bias (56,) = 1624 learned numbers
x1 (32, 28) → hidden (32, 56)
row 31's first 6 hidden numbers: [0.93, -0.39, -3.85, 2.64, 0.87, -3.58]
row 31's hidden numbers below 0: 41 of 56


No row sees another: change one row of the input, here row 30 (the space before p), and only row 30 of the hidden tensor
changes.

In [9]:
changed = x1.clone()
changed[30] = torch.randn(28, generator=torch.Generator().manual_seed(0))  # a different row 30; every other row untouched
moved = (widen(model, changed) - hidden).abs().amax(dim=1)                # [32]: how far each hidden row moved
rows_moved = torch.nonzero(moved > 0).flatten().tolist()
print('rows of the hidden tensor that changed:', rows_moved)
assert rows_moved == [30]

rows of the hidden tensor that changed: [30]


### ✋ Your turn

Attention is different. Make the same change to row 30 of `x0`, the rows *before* attention, run attention again (M3's step, in the cell below), and list the rows of its result that differ from `x1`. Put the list in `answer`.

In [10]:
changed0 = x0.clone()
changed0[30] = torch.randn(28, generator=torch.Generator().manual_seed(0))
after = changed0 + block.attn(block.ln1(changed0)[None])[0]         # attention, as in M3: [32, 28]
answer = None   # TODO: the rows where `after` differs from x1, as a list (as rows_moved does above)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == [30, 31], 'compare row by row: torch.nonzero((after - x1).abs().amax(dim=1) > 0).flatten().tolist()'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [11]:
#@title Show a solution { display-mode: "form" }
changed0 = x0.clone()
changed0[30] = torch.randn(28, generator=torch.Generator().manual_seed(0))
after = changed0 + block.attn(block.ln1(changed0)[None])[0]
answer = torch.nonzero((after - x1).abs().amax(dim=1) > 0).flatten().tolist()
print('rows 30 and 31 change: row 31 looks back at row 30. Rows 0 to 29 cannot see it (the mask).')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == [30, 31], 'compare row by row: torch.nonzero((after - x1).abs().amax(dim=1) > 0).flatten().tolist()'
    print("✓ right:", answer)

rows 30 and 31 change: row 31 looks back at row 30. Rows 0 to 29 cannot see it (the mask).
✓ right: [30, 31]


## M4.2 · The bend: GELU

*Why bend the numbers?*

**Predict first** (the page asks it too, before its clip; the episode's opening asked it first): a step that only multiplies
and adds turns each number into a weighted sum plus a constant. Stack two such steps, one after the other. Do you get anything
one such step could not do?

### M4.2.1 · Picture it

The hidden numbers come from a multiply and add, and another one narrows them. What does such a step do? Picture one made-up
step on one number: multiply by 2, then add 1. In 1 gives out 3; in 2 gives out 5. Plot every input against its output: the
points fall on one straight line. Any step that only multiplies and adds draws a straight line; only its steepness and its
height can change.

### M4.2.2 · Watch it on our line

*(The page's recorded clip, in words.)*

Each of the 56 hidden numbers of every row now passes through a bend called **GELU** (Gaussian Error Linear Unit): GELU(h) =
h × Φ(h), where Φ(h) is the fraction of the normal bell curve that lies below h. Negative numbers shrink toward 0; positive ones
pass almost unchanged. Of row 31's 41 negative hidden numbers, 22 end within 0.05
of 0. Why bend? Two layers that only multiply and add collapse into one such layer; the bend between them lets two layers do
what one cannot.

### M4.2.3 · Read and run the code

The bend is `gelu` in the teaching code: each number h gets a fraction, how much of a bell curve lies below h (near 0 for very
negative h, near 1 for large h), and the result is h times that fraction. The model's own layer, `block.ff[1]`, gives the same
numbers. The cells below print the curve, then row 31 before and after the bend; the "your turn" shows why the bend
matters: without it, the two layers equal one layer of [28×28].

In [12]:
print('    h   GELU(h)')
for h in [-4.0, -3.0, -2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0, 4.0]:
    print(f"{h:5.1f}  {gelu(torch.tensor(h)).item():7.3f}")
curve = torch.linspace(-6, 6, 1201)
low = int(gelu(curve).argmin())
print('the lowest GELU ever goes:', round(gelu(curve)[low].item(), 3), 'at h =', round(curve[low].item(), 2))
assert torch.allclose(gelu(curve), block.ff[1](curve), atol=1e-6)     # the model's own GELU

    h   GELU(h)
 -4.0   -0.000
 -3.0   -0.004
 -2.0   -0.046
 -1.0   -0.159
 -0.5   -0.154
  0.0    0.000
  0.5    0.346
  1.0    0.841
  2.0    1.954
  3.0    2.996
  4.0    4.000
the lowest GELU ever goes: -0.17 at h = -0.75


Now row 31's 56 hidden numbers, before and after the bend. 41 of them are negative before it, and of
those, 22 end up within 0.05 of 0 after it.

In [13]:
bent = gelu(hidden)                                # [32, 56] → [32, 56]
NEAR = 0.05                                    # "near 0": within 0.05 of 0, as the course's record counts it
before, after = hidden[31], bent[31]
print("row 31's 56 hidden numbers, before → after the bend (smallest first):")
order = before.argsort().tolist()
for i in range(0, 56, 4):
    print('    '.join(f"{before[j]:6.2f} → {after[j]:5.2f}" for j in order[i:i + 4]))
negative = int((before < 0).sum())
near_zero = int(((before < 0) & (after.abs() < NEAR)).sum())
print(negative, 'of 56 are negative before the bend;', near_zero, 'of those end up within', NEAR, 'of 0 after it')

RECORDED_GELU_31 = [0.7654, -0.1368, -0.0002, 2.6302, 0.7072, -0.0006, -0.146, -0.0324, -0.0, 0.5105, -0.1045, -0.1544, -0.1621, -0.0941, -0.0, -0.1572, -0.1216, -0.1215, -0.1689, -0.1699, -0.0007, 0.6562, -0.0299, -0.0009, -0.0, -0.0024, -0.0, -0.1382, -0.1672, 1.1938, -0.1477, 1.1086, -0.0083, 1.1578, 3.1177, -0.0, 0.6076, -0.1663, 0.211, -0.0, -0.1699, -0.0001, -0.0018, -0.0, -0.0347, 0.5523, -0.0947, 1.2278, -0.1527, -0.0, 0.9122, -0.1699, -0.0, -0.0297, -0.0, 3.5754]   # the course's record of row 31 after the bend
assert tuple(bent.shape) == (32, 56) and same(bent[31], RECORDED_GELU_31)
assert negative == 41 and near_zero == 22 == int((after.abs() < NEAR).sum())   # no positive one lands near 0

row 31's 56 hidden numbers, before → after the bend (smallest first):
 -9.23 → -0.00     -8.88 → -0.00     -8.05 → -0.00     -7.40 → -0.00
 -7.16 → -0.00     -6.17 → -0.00     -6.01 → -0.00     -5.91 → -0.00
 -5.29 → -0.00     -4.36 → -0.00     -4.20 → -0.00     -3.85 → -0.00
 -3.58 → -0.00     -3.56 → -0.00     -3.48 → -0.00     -3.27 → -0.00
 -3.17 → -0.00     -2.75 → -0.01     -2.21 → -0.03     -2.21 → -0.03
 -2.17 → -0.03     -2.14 → -0.03     -1.55 → -0.09     -1.47 → -0.10
 -1.33 → -0.12     -1.20 → -0.14     -1.13 → -0.15     -1.11 → -0.15
 -1.07 → -0.15     -1.02 → -0.16     -0.96 → -0.16     -0.87 → -0.17
 -0.74 → -0.17     -0.74 → -0.17     -0.74 → -0.17     -0.68 → -0.17
 -0.63 → -0.17     -0.50 → -0.15     -0.39 → -0.14     -0.33 → -0.12
 -0.23 → -0.09      0.33 →  0.21      0.68 →  0.51      0.72 →  0.55
  0.78 →  0.61      0.83 →  0.66      0.87 →  0.71      0.93 →  0.77
  1.06 →  0.91      1.24 →  1.11      1.29 →  1.16      1.32 →  1.19
  1.35 →  1.23      2.64 →  2.63 

Why bend at all? Two layers that only multiply and add collapse into one such layer: widening then narrowing, with nothing in
between, is the same as one [28×28] layer. The bend between them is what lets two layers do what one cannot.

### ✋ Your turn

Check it on the real layers. Without the bend, widen-then-narrow is `narrow(model, widen(model, x1))`. Build the **one** layer it equals: a weight `W = down.weight @ up.weight` ([28×28]) and a bias `b = down.weight @ up.bias + down.bias` ([28]). The cell then puts the largest gap between that one layer and the two in `answer`.

In [14]:
down = block.ff[2]                                 # the narrowing layer: weight [28, 56], bias [28]
W = None   # TODO: down.weight @ up.weight
b = None   # TODO: down.weight @ up.bias + down.bias
answer = None if W is None or b is None else (block.ln2(x1) @ W.T + b - narrow(model, widen(model, x1))).abs().max().item()

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer < 1e-4, 'W = down.weight @ up.weight and b = down.weight @ up.bias + down.bias; the gap should be about 0'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [15]:
#@title Show a solution { display-mode: "form" }
down = block.ff[2]
W = down.weight @ up.weight                        # [28, 56] @ [56, 28] → [28, 28]
b = down.weight @ up.bias + down.bias              # [28]
answer = (block.ln2(x1) @ W.T + b - narrow(model, widen(model, x1))).abs().max().item()
bent_gap = (block.ln2(x1) @ W.T + b - narrow(model, gelu(widen(model, x1)))).abs().max().item()
print('without the bend, the one layer matches the two to within', f"{answer:.1e}")
print('with the bend in between, it misses by up to', round(bent_gap, 2))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer < 1e-4, 'W = down.weight @ up.weight and b = down.weight @ up.bias + down.bias; the gap should be about 0'
    print("✓ right:", answer)

without the bend, the one layer matches the two to within 7.6e-06
with the bend in between, it misses by up to 21.75
✓ right: 7.62939453125e-06


# Section 2 · Add and measure

The block's second half finishes: each row comes back to its own size and is added onto the stream. Then the readout, borrowed
as a probe, measures what the whole block changed.

## M4.3 · Narrow and add

*Why add the result instead of replacing the row?*

**Predict first** (the page asks it too, before its clip): row 31, as one arrow, measures 3.6 after attention. The
feed-forward update for it measures 11.9. After the add, how long is the row: 15.5,
12.4 or 11.6?

### M4.3.1 · Picture it

The hidden numbers shrink back to the row's size; then what happens to the row? A made-up layer narrows 3 hidden numbers back
to 2: 3 × 2 weights and 2 biases, 8 learned numbers. Out comes an update. Say the row is 1, 2 and the update 3, −1: added
number by number, 4, 1. Swapped in instead, the row would become 3, −1, and what it held would be lost. Added, it stays in the
sum.

### M4.3.2 · Watch it on our line

*(The page's recorded clip, in words.)*

The second feed-forward layer narrows each row back from 56 numbers to 28: a [28×56] weight and 28 biases, 1,596
learned numbers. Together the two layers hold 3,220 of the block's 6,580 learned numbers, about half. Their
result is an **update** for each row, and it is **added** onto the stream, not swapped in: `x2 = x1 + update`. This is the
**residual add**: each row's old numbers stay in the sum, and the feed-forward network only writes onto it. Row 31's length is
2.6 at the start, 3.6 after attention, and 11.6 after the feed-forward network.

### M4.3.3 · Read and run the code

Two short functions in the teaching code: `narrow` takes the narrowing layer, a [28×56] weight and 28 biases, and gives each
row its update, 28 numbers; `feed_forward` chains widen, the bend and narrow, then returns `x + update`: added, not swapped in.
The cells below check it against the model's own block, then measure row 31 as an arrow: its update is 11.9 long, yet after
the add the row is only 11.6.

In [16]:
down = block.ff[2]                                 # the narrowing layer
print('narrowing layer: weight', tuple(down.weight.shape), '+ bias', tuple(down.bias.shape), '=', down.weight.numel() + down.bias.numel(), 'learned numbers')
update = narrow(model, bent)                       # [32, 56] → [32, 28]
x2 = feed_forward(model, x1)                       # [32, 28]: x1 + update
print('bent', tuple(bent.shape), '→ update', tuple(update.shape), '→ x2 = x1 + update', tuple(x2.shape))
ffn = sum(p.numel() for p in block.ff.parameters())
print('the feed-forward network:', ffn, 'of the block\'s', sum(p.numel() for p in block.parameters()), 'learned numbers')

assert down.weight.numel() + down.bias.numel() == 1596 and ffn == 3220 and sum(p.numel() for p in block.parameters()) == 6580
assert torch.equal(x2, x1 + update)                               # added, not swapped in
assert torch.allclose(x2, block(x0[None])[0], atol=1e-5)          # the model's own block: attention, then feed-forward
assert tuple(x2.shape) == (32, 28) and same(update[31, -4:], [0.7164, 0.7527, 4.002, 4.1488]) and same(x2[31, -4:], [0.7697, 1.4879, 3.521, 4.8752])

narrowing layer: weight (28, 56) + bias (28,) = 1596 learned numbers
bent (32, 56) → update (32, 28) → x2 = x1 + update (32, 28)
the feed-forward network: 3220 of the block's 6580 learned numbers


Measure row 31 as an arrow: its **length** is the square root of the sum of its 28 squares. It is 2.56 at the
start. Attention's update is 2.43 long, and the row after attention 3.58. The feed-forward update is
11.89 long, and the row after the block 11.60.

In [17]:
length = lambda r: r.norm().item()                # the square root of the sum of its 28 squares
arrows = [('x0, the start', x0[31]), ("attention's update", att[31]), ('x1, after attention', x1[31]),
          ('the feed-forward update', update[31]), ('x2, after the block', x2[31])]
for name, r in arrows:
    print(f"{name:24s} {length(r):6.2f}")
assert same([length(r) for _, r in arrows], [2.5602, 2.4321, 3.5805, 11.8889, 11.597])

x0, the start              2.56
attention's update         2.43
x1, after attention        3.58
the feed-forward update   11.89
x2, after the block       11.60


### ✋ Your turn

Do lengths add? Put row 31's length after attention plus the length of the feed-forward update, rounded to 2 places, in `answer`, and compare it with the row's length after the block.

In [18]:
answer = None   # TODO: round(x1[31].norm().item() + update[31].norm().item(), 2)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == round(x1[31].norm().item() + update[31].norm().item(), 2), 'add the two lengths: x1[31].norm() and update[31].norm()'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [19]:
#@title Show a solution { display-mode: "form" }
answer = round(x1[31].norm().item() + update[31].norm().item(), 2)
cos = torch.cosine_similarity(x1[31], update[31], dim=0).item()
print(answer, 'is more than', round(x2[31].norm().item(), 2), ': arrows add as arrows, not as lengths.')
print('The update points partly against the row (cosine', round(cos, 2), '; below 0 means partly opposite), so part of it cancels.')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == round(x1[31].norm().item() + update[31].norm().item(), 2), 'add the two lengths: x1[31].norm() and update[31].norm()'
    print("✓ right:", answer)

15.47 is more than 11.6 : arrows add as arrows, not as lengths.
The update points partly against the row (cosine -0.23 ; below 0 means partly opposite), so part of it cancels.
✓ right: 15.47


## M4.4 · What the block changed

*What did the block change?*

**Predict first** (the page asks it too, before its clip): run the readout on row 31 right after attention, before the
feed-forward network. What does it read: p, the character the row holds; r, the model's own first
guess; or nothing clear, about 1 in 65 for each?

### M4.4.1 · Picture it

The rows leave the block as 28 numbers each, and nobody can read 28 numbers by eye. The model's last stage can: the readout
turns a row into a probability for every character. So borrow it as a probe, and read the stream at three points: before the
block, after attention, and after the block. The model itself reads out only at the end; reading earlier is a measurement,
like a thermometer dipped in on the way.

### M4.4.2 · Watch it on our line

*(The page's recorded clip, in words.)*

To see what the block did, run the model's last stages (the final LayerNorm, the readout and softmax, which M5 opens) on the
stream at three points: before the block (`x0`), after attention (`x1`) and after the block (`x2`). Before the block, row
31 reads as p itself, with more than 0.999; after attention, still p: 0.989. After
the feed-forward network: r 0.340, a 0.210, l 0.133, o 0.106, e 0.104, the model's own top 5. The row went from saying which character it is to saying which
character comes next.

### M4.4.3 · Read and run the code

The measurement is `readout_at` in the teaching code: the final LayerNorm, then a score for every character, then softmax,
[32×65] probabilities. The first cell reads row 31 at `x0`, `x1` and `x2` and checks each against the
record; at `x2` it is the model's own output. The second does the same for all 32 rows.

In [20]:
probes = {'x0, before the block': x0, 'x1, after attention': x1, 'x2, after the block': x2}
tops = {}
for name, x in probes.items():
    p = readout_at(model, x)[31]                   # [65]: row 31's probabilities, read out at this point
    v, i = p.topk(5)
    tops[name] = [(tok.chars[k], q) for q, k in zip(v.tolist(), i.tolist())]
    print(f"{name:21s}", '   '.join(f"{ch!r} {q:.3f}" for ch, q in tops[name]))

RECORDED = {"x0": [["p", 1.0], ["b", 0.0], ["s", 0.0], ["t", 0.0], ["f", 0.0]], "x1": [["p", 0.989], ["f", 0.01], ["c", 0.0007], ["b", 0.0001], ["S", 0.0001]], "x2": [["r", 0.3403], ["a", 0.2101], ["l", 0.1328], ["o", 0.1061], ["e", 0.1044]]}   # the course's record: the top 5 at each point
for (name, top), rec in zip(tops.items(), RECORDED.values()):
    assert [ch for ch, _ in top] == [ch for ch, _ in rec] and same([q for _, q in top], [q for _, q in rec]), name
assert tops['x0, before the block'][0][1] > 0.999
assert torch.allclose(readout_at(model, x2), torch.softmax(model(ids[None])[0][0], dim=-1), atol=1e-6)   # the model's own output

x0, before the block  'p' 1.000   'b' 0.000   's' 0.000   't' 0.000   'f' 0.000
x1, after attention   'p' 0.989   'f' 0.010   'c' 0.001   'b' 0.000   'S' 0.000
x2, after the block   'r' 0.340   'a' 0.210   'l' 0.133   'o' 0.106   'e' 0.104


The same measurement on all 32 rows: how often is a row's top character its own character, and how often the character that
really comes next? (The true next characters are the line shifted by one, plus the character that follows the line in the play.)

In [21]:
nxt = text[text.find(LINE) + len(LINE)]            # the character that really follows the line in the play
truth = LINE[1:] + nxt                             # row t's true next character
print('after the line comes', repr(nxt))
for name, x in probes.items():
    top = [tok.chars[k] for k in readout_at(model, x).argmax(dim=1).tolist()]
    own = sum(a == b for a, b in zip(top, LINE))
    right = sum(a == b for a, b in zip(top, truth))
    print(f"{name:21s} top is the row's own character in {own:2d} of 32 rows, the true next character in {right:2d}")
assert nxt == tops['x2, after the block'][0][0]

after the line comes 'r'
x0, before the block  top is the row's own character in 31 of 32 rows, the true next character in  0
x1, after attention   top is the row's own character in 22 of 32 rows, the true next character in  0
x2, after the block   top is the row's own character in  0 of 32 rows, the true next character in 14


### ✋ Your turn

Row 25 is the n of “and”. Read it out before the block and after it. Put row 25's top character after the block in `answer`.

In [22]:
before_25 = readout_at(model, x0)[25]              # [65]
after_25 = readout_at(model, x2)[25]               # [65]
answer = None   # TODO: the character with the highest probability in after_25: tok.chars[int(after_25.argmax())]

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == tok.chars[int(readout_at(model, x2)[25].argmax())], 'tok.chars[int(after_25.argmax())]'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [23]:
#@title Show a solution { display-mode: "form" }
before_25, after_25 = readout_at(model, x0)[25], readout_at(model, x2)[25]
print('before the block:', repr(tok.chars[int(before_25.argmax())]), round(before_25.max().item(), 3))
answer = tok.chars[int(after_25.argmax())]
print('after the block: ', repr(answer), round(after_25.max().item(), 3), ': the d of “and”')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == tok.chars[int(readout_at(model, x2)[25].argmax())], 'tok.chars[int(after_25.argmax())]'
    print("✓ right:", answer)

before the block: 'n' 1.0
after the block:  'd' 0.914 : the d of “and”
✓ right: d
